# 🎓 Guide Complet du Portail Bancaire (Pour Débutant)

Bienvenue dans ce cours interactif ! Ce document (Jupyter Notebook) a été généré spécialement pour toi.
Nous allons explorer ensemble **absolument tout ton projet**, ligne par ligne.

Je t'explique les concepts comme si tu n'avais jamais fait de développement web ou d'architecture logicielle auparavant.

## 🛠️ Le Concept : Les Microservices
Ton projet est basé sur l'architecture **Microservices**.
- **Avant (Monolithe)** : C'est comme construire une très grande maison d'un seul bloc. Si tu veux modifier la salle de bain, tu dois bloquer toute la maison et re-construire le toit. Si le code "Authentification" plante, toute la banque plante.
- **Maintenant (Microservices)** : C'est comme un village composé de plusieurs petites maisons indépendantes (Auth-Service, Account-Service, etc.). Si une maison prend feu ou est en rénovation, les autres fonctionnent toujours.
- **Alternative** : L'approche "Monolithe" ou "Serverless" (Fonctions Cloud). Le monolithe est plus simple à démarrer, mais plus difficile à faire évoluer en grosse équipe. Ici, tu as choisi une architecture digne des grandes banques !

---

## 1️⃣ La Base de Données & L'Infrastructure (Docker)
Avant de regarder le code Java ou React, il faut comprendre l'orchestration. Ton projet utilise **Docker**.
Docker, c'est comme des boîtes de transport (conteneurs) pour logiciels. Ça permet que ton code s'exécute exactement pareil sur ton PC et sur n'importe quel serveur dans le monde.


### Fichier : `docker-compose.yml`
Ce fichier est le chef d'orchestre. Il dit à Docker de lancer les bases de données (PostgreSQL), la file d'attente (RabbitMQ), et tes microservices.


In [ ]:
version: '3.8'

services:
  # ============================================
  # DATABASES (One per service)
  # ============================================

  postgres-auth:
    image: postgres:15-alpine
    container_name: postgres-auth
    environment:
      POSTGRES_DB: ${AUTH_DB_NAME:-auth_db}
      POSTGRES_USER: ${DB_USERNAME:-bankadmin}
      POSTGRES_PASSWORD: ${DB_PASSWORD:-postgres}
      PGDATA: /var/lib/postgresql/data/pgdata
    volumes:
      - postgres-auth-data:/var/lib/postgresql/data
      - ./Auth-Service/src/main/resources/schema.sql:/docker-entrypoint-initdb.d/01-init.sql
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U ${DB_USERNAME:-bankadmin}"]
      interval: 10s
      timeout: 5s
      retries: 5
    networks:
      - banking-network

  postgres-account:
    image: postgres:15-alpine
    container_name: postgres-account
    environment:
      POSTGRES_DB: ${ACCOUNT_DB_NAME:-account_db}
      POSTGRES_USER: ${DB_USERNAME:-bankadmin}
      POSTGRES_PASSWORD: ${DB_PASSWORD:-postgres}
      PGDATA: /var/lib/postgresql/data/pgdata
    volumes:
      - postgres-account-data:/var/lib/postgresql/data
      - ./Account-Service/src/main/resources/schema.sql:/docker-entrypoint-initdb.d/01-init.sql
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U ${DB_USERNAME:-bankadmin}"]
      interval: 10s
      timeout: 5s
      retries: 5
    networks:
      - banking-network

  postgres-transaction:
    image: postgres:15-alpine
    container_name: postgres-transaction
    environment:
      POSTGRES_DB: ${TRANSACTION_DB_NAME:-transaction_db}
      POSTGRES_USER: ${DB_USERNAME:-bankadmin}
      POSTGRES_PASSWORD: ${DB_PASSWORD:-postgres}
      PGDATA: /var/lib/postgresql/data/pgdata
    volumes:
      - postgres-transaction-data:/var/lib/postgresql/data
      - ./Transaction-service/src/main/resources/schema.sql:/docker-entrypoint-initdb.d/01-init.sql
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U ${DB_USERNAME:-bankadmin}"]
      interval: 10s
      timeout: 5s
      retries: 5
    networks:
      - banking-network

  # ============================================
  # MESSAGE BROKER
  # ============================================

  rabbitmq:
    image: rabbitmq:3.12-management-alpine
    container_name: rabbitmq
    environment:
      RABBITMQ_DEFAULT_USER: ${RABBITMQ_USERNAME:-guest}
      RABBITMQ_DEFAULT_PASS: ${RABBITMQ_PASSWORD:-guest}
    ports:
      - "15672:15672"  # Management UI
    volumes:
      - rabbitmq-data:/var/lib/rabbitmq
    healthcheck:
      test: ["CMD", "rabbitmq-diagnostics", "-q", "ping"]
      interval: 10s
      timeout: 5s
      retries: 5
    networks:
      - banking-network

  # ============================================
  # MICROSERVICES
  # ============================================

  auth-service:
    build: ./Auth-Service
    container_name: auth-service
    ports:
      - "8081:8081"
    environment:
      SPRING_DATASOURCE_URL: jdbc:postgresql://postgres-auth:5432/${AUTH_DB_NAME:-auth_db}
      SPRING_DATASOURCE_USERNAME: ${DB_USERNAME:-bankadmin}
      SPRING_DATASOURCE_PASSWORD: ${DB_PASSWORD:-postgres}
      SPRING_JPA_HIBERNATE_DDL_AUTO: validate
      SECURITY_JWT_SECRET: ${JWT_SECRET:-your-secret-key-minimum-32-characters-here}
      SECURITY_JWT_EXPIRATION: 3600000
      LOG_LEVEL: ${LOG_LEVEL:-INFO}
    depends_on:
      postgres-auth:
        condition: service_healthy
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8081/api/auth/health"]
      interval: 15s
      timeout: 5s
      retries: 3
      start_period: 30s
    networks:
      - banking-network

  account-service:
    build: ./Account-Service
    container_name: account-service
    ports:
      - "8082:8082"
    environment:
      SPRING_DATASOURCE_URL: jdbc:postgresql://postgres-account:5432/${ACCOUNT_DB_NAME:-account_db}
      SPRING_DATASOURCE_USERNAME: ${DB_USERNAME:-bankadmin}
      SPRING_DATASOURCE_PASSWORD: ${DB_PASSWORD:-postgres}
      SPRING_JPA_HIBERNATE_DDL_AUTO: validate
      SECURITY_JWT_SECRET: ${JWT_SECRET:-your-secret-key-minimum-32-characters-here}
      SECURITY_JWT_EXPIRATION: 3600000
      AUTH_SERVICE_URL: http://auth-service:8081
      LOG_LEVEL: ${LOG_LEVEL:-INFO}
    depends_on:
      postgres-account:
        condition: service_healthy
      auth-service:
        condition: service_healthy
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8082/api/accounts/health"]
      interval: 15s
      timeout: 5s
      retries: 3
      start_period: 30s
    networks:
      - banking-network

  transaction-service:
    build: ./Transaction-Service
    container_name: transaction-service
    ports:
      - "8083:8083"
    environment:
      SPRING_DATASOURCE_URL: jdbc:postgresql://postgres-transaction:5432/${TRANSACTION_DB_NAME:-transaction_db}
      SPRING_DATASOURCE_USERNAME: ${DB_USERNAME:-bankadmin}
      SPRING_DATASOURCE_PASSWORD: ${DB_PASSWORD:-postgres}
      SPRING_JPA_HIBERNATE_DDL_AUTO: validate
      SPRING_RABBITMQ_HOST: ${RABBITMQ_HOST:-rabbitmq}
      SPRING_RABBITMQ_PORT: ${RABBITMQ_PORT:-5672}
      SPRING_RABBITMQ_USERNAME: ${RABBITMQ_USERNAME:-guest}
      SPRING_RABBITMQ_PASSWORD: ${RABBITMQ_PASSWORD:-guest}
      SECURITY_JWT_SECRET: ${JWT_SECRET:-your-secret-key-minimum-32-characters-here}
      SECURITY_JWT_EXPIRATION: 3600000
      AUTH_SERVICE_URL: http://auth-service:8081
      ACCOUNT_SERVICE_URL: http://account-service:8082
      LOG_LEVEL: ${LOG_LEVEL:-INFO}
    depends_on:
      postgres-transaction:
        condition: service_healthy
      rabbitmq:
        condition: service_healthy
      auth-service:
        condition: service_healthy
      account-service:
        condition: service_healthy
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8083/api/transactions/health"]
      interval: 15s
      timeout: 5s
      retries: 3
      start_period: 30s
    networks:
      - banking-network

  notification-service:
    build: ./Notification-Service
    container_name: notification-service
    ports:
      - "8084:8084"
    environment:
      SPRING_RABBITMQ_HOST: ${RABBITMQ_HOST:-rabbitmq}
      SPRING_RABBITMQ_PORT: ${RABBITMQ_PORT:-5672}
      SPRING_RABBITMQ_USERNAME: ${RABBITMQ_USERNAME:-guest}
      SPRING_RABBITMQ_PASSWORD: ${RABBITMQ_PASSWORD:-guest}
      SPRING_MAIL_HOST: ${SMTP_HOST:-smtp.gmail.com}
      SPRING_MAIL_PORT: ${SMTP_PORT:-587}
      SPRING_MAIL_USERNAME: ${SMTP_USERNAME:-your-email@gmail.com}
      SPRING_MAIL_PASSWORD: ${SMTP_PASSWORD:-app-password}
      NOTIFICATION_FROM_EMAIL: ${NOTIFICATION_FROM_EMAIL:-noreply@banking.local}
      LOG_LEVEL: ${LOG_LEVEL:-INFO}
    depends_on:
      rabbitmq:
        condition: service_healthy
    networks:
      - banking-network

  # ============================================
  # API GATEWAY (Nginx)
  # ============================================

  nginx:
    build: ./Nginx
    container_name: nginx-gateway
    ports:
      - "80:80"
      - "443:443"
    volumes:
      - ./Nginx/nginx.conf:/etc/nginx/conf.d/default.conf:ro
      - ./Nginx/ssl:/etc/nginx/ssl:ro
      - ./Front-End/dist:/usr/share/nginx/html:ro
    depends_on:
      auth-service:
        condition: service_healthy
      account-service:
        condition: service_healthy
      transaction-service:
        condition: service_healthy
    networks:
      - banking-network

  # ============================================
  # FRONTEND (served by Nginx)
  # ============================================

  frontend:
    build: ./Front-End
    container_name: frontend
    depends_on:
      - nginx
    networks:
      - banking-network

volumes:
  postgres-auth-data:
  postgres-account-data:
  postgres-transaction-data:
  rabbitmq-data:

networks:
  banking-network:
    driver: bridge



**Explication ligne par ligne de `docker-compose.yml` :**
- `version: '3.8'` : La version de la syntaxe Docker qu'on utilise.
- `services:` : On liste toutes les "machines virtuelles" (conteneurs) qu'on veut allumer.
- `postgres-auth:` : On crée un conteneur pour la base de données PostgreSQL dédiée au service d'authentification.
  - `image: postgres:15-alpine` : On télécharge l'image officielle. `alpine` signifie que c'est une version ultra-légère (pour consommer moins de RAM).
  - `environment:` : Ce sont les variables secrètes (mot de passe, utilisateur) qu'on injecte dans la base de données au démarrage.
  - `volumes:` : On indique à Docker de sauvegarder les données physiques sur ton PC (`postgres-auth-data:/...`), sinon à chaque redémarrage, la banque serait vidée !
- `rabbitmq:` : C'est le "facteur" de notre application. On l'utilise pour que les services se parlent de manière asynchrone (nous verrons cela avec Transaction et Notification).
- `auth-service: build: ./Auth-Service` : Ici on dit "ne télécharge pas ce service sur Internet, regarde dans le dossier `./Auth-Service` et construis-le toi-même" (grâce au fichier Dockerfile).
- `depends_on:` : Spécifie l'ordre de démarrage. `auth-service` attend que `postgres-auth` soit prêt avant de démarrer.

---

## 2️⃣ Service d'Authentification (Auth-Service)
Ce service est le videur de la boîte de nuit. Il vérifie qui tu es, et te donne un "bracelet" (Jeton JWT) pour que tu puisses naviguer dans l'application sans redonner ton mot de passe à chaque fois.

### Le Modèle User (Entité)
C'est la représentation d'un Utilisateur dans la base de données.


In [ ]:
package com.banking.auth.entity;

import jakarta.persistence.*;
import lombok.AllArgsConstructor;
import lombok.Data;
import lombok.NoArgsConstructor;

import java.time.LocalDateTime;

@Entity
@Table(name = "users")
@Data
@NoArgsConstructor
@AllArgsConstructor
public class User {

    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;

    @Column(nullable = false, unique = true)
    private String email;

    @Column(nullable = false)
    private String passwordHash;

    @Column(name = "first_name")
    private String firstName;

    @Column(name = "last_name")
    private String lastName;

    @Column(nullable = false)
    private Boolean isActive = true;

    @Column(name = "created_at", nullable = false, updatable = false)
    private LocalDateTime createdAt = LocalDateTime.now();

    @PrePersist
    protected void onCreate() {
        if (createdAt == null) {
            createdAt = LocalDateTime.now();
        }
    }
}



**Explication ligne par ligne de `User.java` :**
- `@Entity` et `@Table(name = "users")` : Ce sont des annotations Hibernate. Ça dit "S'il te plaît Java, convertis cette classe en une vraie table dans la base de données PostgreSQL qui s'appelle 'users'".
- `@Data`, `@NoArgsConstructor`, `@AllArgsConstructor` : C'est la librairie **Lombok**. Elle génère de manière invisible les "Getters" (ex: `getId()`) et "Setters" (ex: `setId(1)`) pour nous faire gagner du temps et de l'espace visuel dans le code.
- `@Id` et `@GeneratedValue(strategy = GenerationType.IDENTITY)` : `id` est la Clé Primaire de la table (chaque utilisateur a un numéro unique). IDENTITY veut dire que PostgreSQL va compter automatiquement (1, 2, 3...).
- `passwordHash` : REMARQUE IMPORTANTE ! On ne stocke **JAMAIS** un mot de passe en clair. On le "hache" (transforme en une bouillie de caractères illisibles mais vérifiables) avec l'algorithme BCrypt.
- `@PrePersist` : Juste avant de sauvegarder de nouvelles données, cette fonction s'exécute pour inscrire la date et l'heure exactes de création.

### Comment on génère le JWT (Bracelet d'accès) ?


In [ ]:
package com.banking.auth.util;

import io.jsonwebtoken.*;
import io.jsonwebtoken.security.Keys;
import lombok.extern.slf4j.Slf4j;
import org.springframework.beans.factory.annotation.Value;
import org.springframework.stereotype.Component;

import javax.crypto.SecretKey;
import java.nio.charset.StandardCharsets;
import java.util.Date;
import java.util.HashMap;
import java.util.Map;

@Component
@Slf4j
public class JwtProvider {

    @Value("${security.jwt.secret}")
    private String jwtSecret;

    @Value("${security.jwt.expiration:3600000}")
    private long jwtExpirationMs;

    public String generateToken(Long userId, String email) {
        Map<String, Object> claims = new HashMap<>();
        claims.put("userId", userId);
        claims.put("email", email);

        Date now = new Date();
        Date expiryDate = new Date(now.getTime() + jwtExpirationMs);

        SecretKey key = Keys.hmacShaKeyFor(jwtSecret.getBytes(StandardCharsets.UTF_8));

        return Jwts.builder()
                .claims(claims)
                .subject(String.valueOf(userId))
                .issuedAt(now)
                .expiration(expiryDate)
                .signWith(key, SignatureAlgorithm.HS512)
                .compact();
    }

    public Long getUserIdFromToken(String token) {
        try {
            SecretKey key = Keys.hmacShaKeyFor(jwtSecret.getBytes(StandardCharsets.UTF_8));
            Claims claims = Jwts.parserBuilder()
                    .setSigningKey(key)
                    .build()
                    .parseClaimsJws(token)
                    .getBody();

            return Long.valueOf(claims.getSubject());
        } catch (JwtException | NumberFormatException e) {
            log.error("Échec de l'extraction de l'ID utilisateur du jeton", e);
            return null;
        }
    }

    public String getEmailFromToken(String token) {
        try {
            SecretKey key = Keys.hmacShaKeyFor(jwtSecret.getBytes(StandardCharsets.UTF_8));
            Claims claims = Jwts.parserBuilder()
                    .setSigningKey(key)
                    .build()
                    .parseClaimsJws(token)
                    .getBody();

            return (String) claims.get("email");
        } catch (JwtException e) {
            log.error("Échec de l'extraction de l'email du jeton", e);
            return null;
        }
    }

    public boolean validateToken(String token) {
        try {
            SecretKey key = Keys.hmacShaKeyFor(jwtSecret.getBytes(StandardCharsets.UTF_8));
            Jwts.parserBuilder()
                    .setSigningKey(key)
                    .build()
                    .parseClaimsJws(token);
            return true;
        } catch (MalformedJwtException e) {
            log.error("Format de jeton JWT invalide", e);
        } catch (ExpiredJwtException e) {
            log.error("Le jeton JWT a expiré", e);
        } catch (UnsupportedJwtException e) {
            log.error("Le jeton JWT n'est pas pris en charge", e);
        } catch (IllegalArgumentException e) {
            log.error("La chaîne de réclamations JWT est vide", e);
        } catch (SignatureException e) {
            log.error("La validation de la signature JWT a échoué", e);
        } catch (JwtException e) {
            log.error("La validation du JWT a échoué", e);
        }
        return false;
    }

    public boolean isTokenExpired(String token) {
        try {
            SecretKey key = Keys.hmacShaKeyFor(jwtSecret.getBytes(StandardCharsets.UTF_8));
            Claims claims = Jwts.parserBuilder()
                    .setSigningKey(key)
                    .build()
                    .parseClaimsJws(token)
                    .getBody();

            return claims.getExpiration().before(new Date());
        } catch (ExpiredJwtException e) {
            return true;
        } catch (JwtException e) {
            log.error("Échec de la vérification de l'expiration du jeton", e);
            return true;
        }
    }

    public long getExpirationTime() {
        return jwtExpirationMs;
    }
}



**Explication de `JwtProvider.java` :**
- Le JWT (JSON Web Token) est une carte d'identité numérique. 
- Au lieu que le serveur se souvienne de tout le monde (ce qui consomme de la mémoire RAM), on encode un mot de passe secret (`jwtSecret`) avec le nom de l'utilisateur. Le navigateur garde cette carte.
- `Jwts.builder().claims(claims).subject...` : On construit le token. On y place `userId`, `email` et une `expiration` (durée de validité, souvent 1 heure). Ainsi, si le token est volé, il expirera vite.
- `validateToken` : Quand un utilisateur demande à voir son compte bancaire, on prend son Jeton et on vérifie si la signature correspond bien à notre `jwtSecret`.

---

## 3️⃣ Compte de l'utilisateur (Account-Service)
Celui-ci est le "guichetier". Il gère l'argent, le type de compte (Courant, Épargne) et récupère l'information de manière sécurisée en analysant le jeton JWT.

### Le Contrôleur (AccountController)


In [ ]:
package com.banking.account.controller;

import com.banking.account.dto.AccountDTO;
import com.banking.account.service.AccountService;
import com.banking.account.util.JwtExtractor;
import lombok.RequiredArgsConstructor;
import lombok.extern.slf4j.Slf4j;
import org.springframework.http.HttpStatus;
import org.springframework.http.ResponseEntity;
import org.springframework.web.bind.annotation.*;

import jakarta.servlet.http.HttpServletRequest;
import java.util.List;

@RestController
@RequestMapping("/api/accounts")
@RequiredArgsConstructor
@Slf4j
public class AccountController {

    private final AccountService accountService;
    private final JwtExtractor jwtExtractor;

    @GetMapping
    public ResponseEntity<List<AccountDTO>> getUserAccounts(HttpServletRequest request) {
        try {
            Long userId = jwtExtractor.extractUserId(request);
            List<AccountDTO> accounts = accountService.getUserAccounts(userId);
            return ResponseEntity.ok(accounts);
        } catch (Exception e) {
            log.error("Error fetching accounts: {}", e.getMessage());
            return ResponseEntity.status(HttpStatus.UNAUTHORIZED).build();
        }
    }

    @GetMapping("/{id}")
    public ResponseEntity<AccountDTO> getAccount(@PathVariable Long id, HttpServletRequest request) {
        try {
            Long userId = jwtExtractor.extractUserId(request);
            accountService.verifyAccountOwnership(id, userId);
            AccountDTO account = accountService.getAccount(id, userId);
            return ResponseEntity.ok(account);
        } catch (Exception e) {
            log.error("Error fetching account: {}", e.getMessage());
            return ResponseEntity.status(HttpStatus.FORBIDDEN).build();
        }
    }

    @GetMapping("/health")
    public ResponseEntity<String> health() {
        return ResponseEntity.ok("Account Service is healthy");
    }
}



**Explication de `AccountController.java` :**
- Le "Contrôleur" (Controller) est le point d'entrée d'Internet. C'est l'URL que le code React (frontend) va appeler.
- `@RestController` et `@RequestMapping("/api/accounts")` : Si le frontend fait une requête HTTP sur `http://serveur.com/api/accounts`, c'est cette classe Java qui va répondre.
- `@GetMapping` : Ce bloc s'active quand le serveur reçoit une méthode HTTP de type "GET" (demander de l'information).
- `Long userId = jwtExtractor.extractUserId(request);` : **Très important**. Au lieu de demander 'De quel compte veux-tu voir la liasse de billets ?', on regarde le token JWT (le bracelet dont on a parlé). Ainsi un utilisateur ne peut pas tricher et voir le compte de quelqu'un d'autre !
- `ResponseEntity.status(HttpStatus.FORBIDDEN).build()` : Si quelqu'un de mal intentionné essaye d'accéder au compte n°5 alors qu'il est l'utilisateur n°2, on lance une erreur `FORBIDDEN` (403 - Accès Interdit).

---

## 4️⃣ Faire des Virements (Transaction Service & RabbitMQ)
Ici nous abordons l'un des concepts les plus puissants : l'**Événementiel (Event-Driven Architecture)** avec RabbitMQ.

Quand tu fais un virement, 2 choses doivent se passer :
1. Déplacer l'argent (Transaction-Service)
2. Envoyer un email de confirmation (Notification-Service)

*Problème* : L'envoi d'email prend du temps (environ 2-3 secondes), ou peut crasher si le serveur Gmail est en panne.
*Solution* : Le Transaction-Service fait le virement en base de données, et "crie dans une file d'attente (RabbitMQ)" : *"Hé, j'ai fini le virement n°45 !"*. Ensuite il répond `OK` au client web *immédiatement*.
Dans son coin, le Notification-Service entend ce cri, le met dans sa to-do list, et envoie l'email quand il a le temps. C'est de l'**Asynchrone**.

### Comment le Transaction-Service crie (Producer) ?


In [ ]:
package com.banking.transaction.producer;

import com.banking.transaction.event.TransferEvent;
import lombok.RequiredArgsConstructor;
import lombok.extern.slf4j.Slf4j;
import org.springframework.amqp.rabbit.core.RabbitTemplate;
import org.springframework.stereotype.Component;

@Component
@RequiredArgsConstructor
@Slf4j
public class TransferEventProducer {

    private final RabbitTemplate rabbitTemplate;

    public void publishTransferCompleted(TransferEvent event) {
        try {
            rabbitTemplate.convertAndSend(
                    "transfers-exchange",
                    "transfer.completed",
                    event
            );
            log.info("Transfer event published for transaction {}", event.getTransactionId());
        } catch (Exception e) {
            log.error("Failed to publish transfer event: {}", e.getMessage(), e);
        }
    }
}



**Explication :**
- `rabbitTemplate.convertAndSend("transfers-exchange", "transfer.completed", event);`
- La `rabbitTemplate` est l'outil Spring qui parle à RabbitMQ. On envoie l'objet `event` (le reçu du virement) sur un canal appelé `transfers-exchange`.
- On ne se soucie pas de savoir SI le Notification Service est allumé ou éteint ! Si le Notification-Service est éteint, RabbitMQ garde le message au chaud. Quand on allumera Notification-Service, il lira tous les messages en attente. C'est ce qu'on appelle "le découplage".

### Comment le Notification-Service écoute (Listener) ?


In [ ]:
package com.banking.notification.listener;

import com.banking.notification.event.TransferEvent;
import com.banking.notification.service.EmailService;
import lombok.RequiredArgsConstructor;
import lombok.extern.slf4j.Slf4j;
import org.springframework.amqp.rabbit.annotation.RabbitListener;
import org.springframework.stereotype.Component;

@Component
@RequiredArgsConstructor
@Slf4j
public class TransferEventListener {

    private final EmailService emailService;

    @RabbitListener(queues = "transfer-completed-queue")
    public void handleTransferCompleted(TransferEvent event) {
        try {
            log.info("Received transfer event for transaction: {}", event.getTransactionId());

            // Note: In a real system, you would fetch user details from Auth Service
            // For now, just log the event
            String fromUserEmail = "from_user_" + event.getFromUserId() + "@example.com";
            String toUserEmail = "to_user_" + event.getToUserId() + "@example.com";
            String fromUserName = "User " + event.getFromUserId();
            String toUserName = "User " + event.getToUserId();

            // Send email to both parties
            emailService.sendTransferNotification(
                    fromUserEmail,
                    fromUserName,
                    event.getAmount().toString(),
                    "Outgoing transfer: " + event.getDescription()
            );

            emailService.sendTransferNotification(
                    toUserEmail,
                    toUserName,
                    event.getAmount().toString(),
                    "Incoming transfer: " + event.getDescription()
            );

            log.info("Transfer notifications sent for transaction {}", event.getTransactionId());
        } catch (Exception e) {
            log.error("Error processing transfer event: {}", e.getMessage(), e);
            // In production, you might want to implement retry logic here
        }
    }
}



**Explication :**
- `@RabbitListener(queues = "transfer-completed-queue")` : C'est comme s'abonner à une station de radio. Dès qu'un message arrive dans la file (`queue`) "transfer-completed", cette fonction tourne instantanément, toute seule.
- Une fois le message reçu, on appelle `emailService.sendTransferNotification()`. Ce composant est juste là pour écouter et déclencher la suite.

---

## 5️⃣ L'Interface Utilisateur (Front-End React)
Le Front-End est ce que l'utilisateur voit dans son navigateur (Chrome, Firefox). Ton projet utilise **React** et **Vite**.
- **React** permet de construire des composants réutilisables (des boutons, des formulaires) qui changent sans avoir à recharger toute la page web (c'est ce qu'on appelle une SPA : Single Page Application).
- **Vite** est l'outil ultra-rapide qui va compiler notre code brouillon pour que les navigateurs le comprennent (du Javascript, de l'HTML, du CSS minimisé).

### L'Appel au Backend : `client.js`


In [ ]:
const baseUrl = import.meta.env.VITE_API_BASE_URL || '';

function request(path, options = {}) {
  const url = `${baseUrl}${path}`;
  return fetch(url, {
    ...options,
    credentials: 'include',
    headers: {
      'Content-Type': 'application/json',
      ...options.headers,
    },
  });
}

export async function login(username, password) {
  const res = await request('/api/auth/login', {
    method: 'POST',
    body: JSON.stringify({ username, password }),
  });
  if (!res.ok) {
    const data = await res.json().catch(() => ({}));
    throw new Error(data.error || 'Échec de la connexion');
  }
  return res.json();
}

export async function logout() {
  await request('/api/auth/logout', { method: 'POST' });
}

export async function getMe() {
  const res = await request('/api/auth/me');
  if (!res.ok) return null;
  return res.json();
}

export async function getAccounts() {
  const res = await request('/api/accounts');
  if (!res.ok) throw new Error('Impossible de charger les comptes');
  return res.json();
}

export async function getAccount(id) {
  const res = await request(`/api/accounts/${id}`);
  if (!res.ok) return null;
  return res.json();
}

export async function getTransactions(accountId) {
  const res = await request(`/api/accounts/${accountId}/transactions`);
  if (!res.ok) throw new Error('Impossible de charger les transactions');
  return res.json();
}



**Explication :**
- `function request(path, options)` : C'est notre fonction générique pour contacter nos microservices. `fetch` est la fonction standard du navigateur pour faire un appel HTTP.
- Pourquoi `credentials: 'include'` ? Ça sert à transporter automatiquement les cookies et les autorisations (le fameux jeton JWT) à chaque roquette.
- Toutes nos fonctions (`login`, `getAccounts`) utilisent cette fonction. Par exemple, si on n'arrive pas à charger les comptes (`!res.ok`), on jette une erreur "Impossible de charger les comptes" qui sera affichée en rouge par React à l'écran.

### L'Interface React : La Page `Accounts.jsx`


In [ ]:
import { useState, useEffect } from 'react'
import { Link, useNavigate } from 'react-router-dom'
import { getAccounts, logout } from '../api/client'
import './Accounts.css'

export default function Accounts() {
  const [accounts, setAccounts] = useState([])
  const [loading, setLoading] = useState(true)
  const [error, setError] = useState('')
  const navigate = useNavigate()

  useEffect(() => {
    getAccounts()
      .then(setAccounts)
      .catch((err) => setError(err.message))
      .finally(() => setLoading(false))
  }, [])

  async function handleLogout() {
    await logout()
    navigate('/login', { replace: true })
  }

  if (loading) return <div className="page-loading">Chargement des comptes…</div>
  if (error) return <div className="page-error">{error}</div>

  return (
    <div className="accounts-page">
      <header className="app-header">
        <h1>Mes comptes</h1>
        <button type="button" className="logout-btn" onClick={handleLogout}>
          Déconnexion
        </button>
      </header>
      <main className="accounts-list">
        {accounts.length === 0 ? (
          <p className="empty">Aucun compte.</p>
        ) : (
          <ul>
            {accounts.map((acc) => (
              <li key={acc.id}>
                <Link to={`/accounts/${acc.id}`} className="account-card">
                  <span className="account-label">{acc.label}</span>
                  <span className="account-iban">{acc.iban}</span>
                  <span className="account-balance">
                    {formatMoney(acc.balance)} {acc.currency}
                  </span>
                </Link>
              </li>
            ))}
          </ul>
        )}
      </main>
    </div>
  )
}

function formatMoney(value) {
  return new Intl.NumberFormat('fr-FR', {
    minimumFractionDigits: 2,
    maximumFractionDigits: 2,
  }).format(value)
}



**Explication de React :**
- `useState([])` : React mémorise des choses dans des "états" (states). Au début, on dit que la liste `accounts` est un tableau vide `[]`. S'il y a une erreur, on la garde dans `error`. Si ça charge, `loading` est vrai ou faux.
- `useEffect` : Ce bloc de code se lance **une et une seule fois** quand la page s'affiche. Que fait-il ? Il appelle `getAccounts()` de notre backend Java, et remplace le tableau vide `[]` par la liste obtenue.
- **Affichage HTML conditionnel** : 
  - `if (loading) return <div>Chargement...</div>` => Tant qu'on attend la réponse du Java, l'utilisateur voit un texte élégant au lieu d'une page blanche !
  - `{accounts.map(...)}` : C'est une boucle. Pour chaque compte bancaire, on crée une `<li>` avec une "Carte Bancaire" en HTML (`account-card`). React va cloner ce bloc HTML autant de fois qu'on possède de comptes.

---

## 6️⃣ Le Chef de la Circulation : Nginx (API Gateway)
Nginx tourne sur le port `80` et `443` (HTTP et HTTPS standards). Il a deux boulots cruciaux :
1. Servir le code compilé HTML/CSS de React aux gens qui visitent ton URL de base.
2. Servir de **Reverse-Proxy (API Gateway)**.

### Pourquoi Nginx pour les API ?
L'utilisateur final n'a pas accès direct à `auth-service` (sur le port `8081`) ! Tous les ports des microservices sont bloqués par les pare-feux (firewalls) dans le monde de l'entreprise. 
Le navigateur contacte Nginx sur `/api/auth/` ; Nginx vérifie discrètement la limitation de débit (pour bloquer les attaques DDoS), et si tout est bon, il achemine la requête lui-même au port 8081. L'utilisateur en est totalement ignorant, c'est transparent et plus sécurisé.

### Regardons la configuration Nginx


In [ ]:
# Déclarations upstream pour les microservices
upstream auth_service {
    server auth-service:8081;
}

upstream account_service {
    server account-service:8082;
}

upstream transaction_service {
    server transaction-service:8083;
}

# Limitation de débit (5 requêtes par seconde par IP pour login)
limit_req_zone $binary_remote_addr zone=login_limit:10m rate=5r/s;
# Limitation de débit générale pour API (100 requêtes par seconde)
limit_req_zone $binary_remote_addr zone=api_limit:10m rate=100r/s;

# Bloc serveur HTTPS - Passerelle API sécurisée
server {
    listen 443 ssl http2;
    server_name localhost;

    # Configuration SSL/TLS
    ssl_certificate /etc/nginx/ssl/cert.pem;
    ssl_certificate_key /etc/nginx/ssl/key.pem;
    ssl_protocols TLSv1.3 TLSv1.2;
    ssl_ciphers HIGH:!aNULL:!MD5;
    ssl_prefer_server_ciphers on;
    ssl_session_cache shared:SSL:10m;
    ssl_session_timeout 10m;

    # En-têtes de sécurité
    add_header Strict-Transport-Security "max-age=31536000; includeSubDomains" always;
    add_header X-Content-Type-Options "nosniff" always;
    add_header X-Frame-Options "DENY" always;
    add_header X-XSS-Protection "1; mode=block" always;
    add_header Referrer-Policy "strict-origin-when-cross-origin" always;

    # En-têtes CORS (pour éviter les problèmes cross-origin)
    add_header Access-Control-Allow-Origin "https://localhost" always;
    add_header Access-Control-Allow-Methods "GET, POST, PUT, DELETE, OPTIONS" always;
    add_header Access-Control-Allow-Headers "DNT, User-Agent, X-Requested-With, If-Modified-Since, Cache-Control, Content-Type, Range, Authorization" always;

    # Gestion des requêtes OPTIONS (préflight CORS)
    if ($request_method = 'OPTIONS') {
        return 204;
    }

    # ============================================
    # ENDPOINT HEALTH CHECK
    # ============================================
    location /health {
        access_log off;
        return 200 "Gateway is healthy\n";
        add_header Content-Type text/plain;
    }

    # ============================================
    # FICHIERS STATIQUES (Frontend React)
    # ============================================
    location / {
        root /usr/share/nginx/html;
        try_files $uri $uri/ /index.html;

        # Contrôle cache pour index.html
        location = /index.html {
            add_header Cache-Control "public, max-age=0, must-revalidate";
        }

        # Cache des assets statiques (JS, CSS, images)
        location ~* \.(js|css|png|jpg|jpeg|gif|ico|svg|woff|woff2|ttf|eot)$ {
            add_header Cache-Control "public, max-age=31536000, immutable";
        }
    }

    # ============================================
    # SERVICE D'AUTHENTIFICATION
    # ============================================
    location /api/auth/ {
        # Limiter le taux de requêtes
        limit_req zone=api_limit burst=10 nodelay;

        # Proxy vers Auth Service
        proxy_pass http://auth_service;
        proxy_http_version 1.1;

        # Transférer les en-têtes vers le backend
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;
        proxy_set_header X-Forwarded-Host $server_name;

        # Délais d'expiration de connexion
        proxy_connect_timeout 60s;
        proxy_send_timeout 60s;
        proxy_read_timeout 60s;

        # Configuration du buffering
        proxy_buffering on;
        proxy_buffer_size 4k;
        proxy_buffers 8 4k;
    }

    # ============================================
    # SERVICE DE GESTION DES COMPTES
    # ============================================
    location /api/accounts/ {
        # Limiter le taux de requêtes
        limit_req zone=api_limit burst=20 nodelay;

        # Proxy vers Account Service
        proxy_pass http://account_service;
        proxy_http_version 1.1;

        # Transférer les en-têtes
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;
        proxy_set_header X-Forwarded-Host $server_name;
        # Important: Transférer le header Authorization (JWT token)
        proxy_set_header Authorization $http_authorization;

        # Délais d'expiration
        proxy_connect_timeout 60s;
        proxy_send_timeout 60s;
        proxy_read_timeout 60s;

        # Buffering
        proxy_buffering on;
        proxy_buffer_size 4k;
        proxy_buffers 8 4k;
    }

    # ============================================
    # SERVICE DE TRANSACTIONS
    # ============================================
    location /api/transactions/ {
        # Limiter le taux de requêtes
        limit_req zone=api_limit burst=15 nodelay;

        # Proxy vers Transaction Service
        proxy_pass http://transaction_service;
        proxy_http_version 1.1;

        # Transférer les en-têtes
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;
        proxy_set_header X-Forwarded-Host $server_name;
        # Important: Transférer le header Authorization (JWT token)
        proxy_set_header Authorization $http_authorization;

        # Délais d'expiration
        proxy_connect_timeout 60s;
        proxy_send_timeout 60s;
        proxy_read_timeout 60s;

        # Buffering
        proxy_buffering on;
        proxy_buffer_size 4k;
        proxy_buffers 8 4k;
    }

    # ============================================
    # GESTION DES ERREURS
    # ============================================
    error_page 400 /errors/400.json;
    error_page 401 /errors/401.json;
    error_page 403 /errors/403.json;
    error_page 404 /errors/404.json;
    error_page 500 502 503 504 /errors/500.json;

    location /errors/ {
        internal;
        default_type application/json;
        root /usr/share/nginx/html;
    }

    # ============================================
    # DENY ACCESS AUX CHEMINS SENSIBLES
    # ============================================
    # Bloquer l'accès aux fichiers cachés (.*)
    location ~ /\. {
        deny all;
        access_log off;
        log_not_found off;
    }

    # Bloquer les fichiers de backup (~)
    location ~ ~$ {
        deny all;
        access_log off;
        log_not_found off;
    }
}

# ============================================
# BLOC SERVEUR HTTP - REDIRECTION HTTPS
# ============================================
server {
    listen 80;
    server_name _;

    # Rediriger toutes les requêtes HTTP vers HTTPS
    return 301 https://$host$request_uri;
}



**Explication de `nginx.conf` :**
- `upstream auth_service { server auth-service:8081; }` : Ici on indique à Nginx où sont les vrais serveurs dans le réseau Docker. Si plus tard on a 3 serveurs auth-service pour diviser la charge, on les mettrait tous ici !
- `limit_req_zone ... rate=5r/s;` : **SÉCURITÉ**. On autorise un maximum de 5 requêtes par seconde par adresse IP (le client). Quelqu'un qui essaye 1000 mots de passe par seconde sera bloqué automatiquement !
- `location /api/auth/ { proxy_pass http://auth_service; }` : C'est la règle de routage (Reverse Proxy). Chaque url `/api/auth` arrive dans la machine Java `8081`. 
- `proxy_set_header Authorization $http_authorization;` : **Essentiel**. Sans cette ligne, Nginx filtrerait et détruirait notre précieux Jeton JWT lors du transfert de la requête de Chrome vers Java.

---

## Conclusion
Tu viens de comprendre tous les morceaux de ton projet !
1. **React** (Le bouton que le client pique).
2. **Nginx** (Le routeur de sécurité).
3. **Auth/Account Services (Spring Boot)** (Les cerveaux métiers).
4. **PostgreSQL** (La mémoire long-terme).
5. **RabbitMQ / Notification Service** (Le système neuronal asynchrone).
6. **Docker** (L'univers qui maintient toutes ces planètes en orbite).

*Tu peux taper `Run` ou fermer ce notebook.* Garde-le précieusement 💎 !

